In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('✅ Ready!')

✅ Ready!


In [4]:
from google.colab import drive
drive.mount('/content/drive')

EPICLIM_PATH = '/content/drive/MyDrive/DiseaseSpread/dataset/raw_data/epiclim_data.csv'

df_raw = pd.read_csv(EPICLIM_PATH)
print(f'✅ Loaded EpiClim | Shape: {df_raw.shape}')
df_raw.head(3)

Mounted at /content/drive
✅ Loaded EpiClim | Shape: (8985, 15)


,Unnamed: 0,week_of_outbreak,state_ut,district,Disease,Cases,Deaths,day,mon,year,Latitude,Longitude,preci,LAI,Temp
0,0,1st week,Meghalaya,East Jaintia Hills,Acute Diarrhoeal Disease,160,NaN,2,1,2022,25.251576,92.484050,0.020354,34.5,291.533333
1,1,2nd week,Maharashtra,Gadchiroli,Malaria,7,2.0,10,1,2022,19.759070,80.162281,0.007479,9.0,299.970000
2,2,3rd week,Tamil Nadu,Pudukottai,Acute Diarrhoeal Disease,8,NaN,18,1,2022,10.382651,78.819126,0.107413,12.0,300.766667


In [5]:
df = df_raw[df_raw['state_ut'].str.contains('Tamil', case=False, na=False)].copy()
df = df.reset_index(drop=True)

print(f'Tamil Nadu records : {len(df)}')
print(f'Districts found    : {df["district"].nunique()}')
print(f'Districts list     : {sorted(df["district"].unique())}')

Tamil Nadu records : 735
Districts found    : 65
Districts list     : ['Ariyalur', 'Ariyalur (Perambalur)', 'Coimbatore', 'Coimbatore (Tiruppur)', 'Cuddalore', 'Dharmapuri', 'Dindigul', 'Dindigul - Palani', 'Dindigul-Palani', 'Erode', 'Kallakurichi', 'Kancheepuram', 'Kanyakumari', 'Karur', 'Krishnagiri', 'Madurai', 'Nagapattinam', 'Nagercoil', 'Namakkal', 'Perambalur', 'Pudukkottai', 'Pudukottai', 'Ramanathapuram', 'Ramanathapuram Paramakudi', 'Ramanathapuram ��������������������������������������������������������������������������������������������������������������������������������������������������������������������������������������������������������������������������������������������������� Paramakudi', 'Ramanathapuram- Paramakudi', 'Ramanathapuram-Paramakudi', 'Saidapet', 'Salem', 'Sivaganga', 'Tenkasi', 'Tenkasi (Tirunelveli)', 'Thanjavur', 'The Nilgiris', 'Theni', 'Thiruvallur', 'Thiruvalluvar', 'Thiruvannamalai', 'Thiruvarur', 'Thoothukudi', 'Tiruchirapalli', 'Tiruchirappal

In [6]:
# Convert Temperature: Kelvin → Celsius
df['temp_celsius'] = (df['Temp'] - 273.15).round(2)

# Rename columns for clarity
df = df.rename(columns={
    'state_ut'        : 'state',
    'week_of_outbreak': 'week_label',
    'Disease'         : 'disease',
    'Cases'           : 'cases',
    'Deaths'          : 'deaths',
    'Latitude'        : 'latitude',
    'Longitude'       : 'longitude',
    'preci'           : 'rainfall_mm',
    'LAI'             : 'leaf_area_index',
})

# Create proper date column
df['date'] = pd.to_datetime(
    df['year'].astype(str) + '-' +
    df['mon'].astype(str).str.zfill(2) + '-' +
    df['day'].astype(str).str.zfill(2),
    errors='coerce'
)

print('✅ Columns after renaming:')
print(df.columns.tolist())
print(f'\nTemp range: {df["temp_celsius"].min():.1f}°C to {df["temp_celsius"].max():.1f}°C')

✅ Columns after renaming:
['Unnamed: 0', 'week_label', 'state', 'district', 'disease', 'cases', 'deaths', 'day', 'mon', 'year', 'latitude', 'longitude', 'rainfall_mm', 'leaf_area_index', 'Temp', 'temp_celsius', 'date']

Temp range: 12.3°C to 45.6°C


In [7]:
print('Missing values BEFORE:')
print(df.isnull().sum()[df.isnull().sum() > 0])

# Convert 'cases' to numeric, coercing errors to NaN
df['cases'] = pd.to_numeric(df['cases'], errors='coerce')

# Fill missing cases with district+disease median
df['cases'] = df.groupby(['district','disease'])['cases'].transform(
    lambda x: x.fillna(x.median())
)
# Fill missing deaths with 0
df['deaths'] = df['deaths'].fillna(0)
# Fill missing climate with column median
df['rainfall_mm']      = df['rainfall_mm'].fillna(df['rainfall_mm'].median())
df['temp_celsius']     = df['temp_celsius'].fillna(df['temp_celsius'].median())
df['leaf_area_index']  = df['leaf_area_index'].fillna(df['leaf_area_index'].median())

print('\nMissing values AFTER:')
remaining = df.isnull().sum()[df.isnull().sum() > 0]
print(remaining if len(remaining) > 0 else '✅ No missing values!')

Missing values BEFORE:
deaths             573
rainfall_mm         46
leaf_area_index    170
Temp                32
temp_celsius        32
dtype: int64

Missing values AFTER:
Temp    32
dtype: int64


In [8]:
# Census 2011 — Tamil Nadu district population & area
# Source: Office of the Registrar General & Census Commissioner, India
census_2011 = {
    'district': [
        'Chennai','Coimbatore','Madurai','Tiruchirappalli','Salem',
        'Tirunelveli','Vellore','Erode','Thanjavur','Dindigul',
        'Tiruppur','Kanchipuram','Krishnagiri','Cuddalore','Nagapattinam',
        'Namakkal','Theni','Villupuram','The Nilgiris','Pudukkottai',
        'Ramanathapuram','Sivaganga','Thoothukudi','Tiruvannamalai','Virudhunagar'
    ],
    'population': [
        7088000, 3458127, 3038252, 2722290, 3482056,
        3072880, 3927780, 2251744, 2405890, 2159775,
        1795974, 3998252, 1879809, 2605914, 1616450,
        1726601, 1243003, 3458873, 735394, 1618345,
        1353445, 1339101, 1750176, 2464875, 1942288
    ],
    'area_sq_km': [
        426, 4723, 3741, 4404, 5245,
        6823, 6077, 5714, 3481, 6267,
        5189, 4432, 5051, 3703, 2715,
        3363, 3242, 7194, 2545, 4663,
        4175, 4189, 4621, 6191, 4283
    ]
}

df_census = pd.DataFrame(census_2011)
df_census['density_per_sqkm'] = (df_census['population'] / df_census['area_sq_km']).round(2)

print('✅ Census 2011 data ready!')
df_census.head()

✅ Census 2011 data ready!


,district,population,area_sq_km,density_per_sqkm
0,Chennai,7088000,426,16638.50
1,Coimbatore,3458127,4723,732.19
2,Madurai,3038252,3741,812.15
3,Tiruchirappalli,2722290,4404,618.14
4,Salem,3482056,5245,663.88


In [9]:
# Standardize district names for matching (lowercase, strip spaces)
df['district_key']         = df['district'].str.strip().str.lower()
df_census['district_key']  = df_census['district'].str.strip().str.lower()

df_merged = df.merge(
    df_census[['district_key','population','area_sq_km','density_per_sqkm']],
    on='district_key',
    how='left'
)

# Fill unmatched districts with state median density
median_density = df_census['density_per_sqkm'].median()
df_merged['density_per_sqkm'] = df_merged['density_per_sqkm'].fillna(median_density)
df_merged['population']        = df_merged['population'].fillna(df_census['population'].median())

print(f'✅ Merged shape: {df_merged.shape}')
print(f'Unmatched districts: {df_merged["density_per_sqkm"].isnull().sum()}')

✅ Merged shape: (735, 21)
Unmatched districts: 0


In [10]:
df_merged['cases_per_100k'] = (
    df_merged['cases'] / df_merged['population'] * 100_000
).round(4)

print('✅ Normalized cases per 100K added')
df_merged[['district','cases','population','cases_per_100k']].head(6)

✅ Normalized cases per 100K added


,district,cases,population,cases_per_100k
0,Pudukottai,8,2251744.0,0.3553
1,Pudukottai,8,2251744.0,0.3553
2,Ariyalur,6,2251744.0,0.2665
3,Ariyalur,15,2251744.0,0.6662
4,Virudhunagar,17,1942288.0,0.8753
5,Ariyalur,14,2251744.0,0.6217


In [11]:
# Season based on month
def get_season(month):
    if month in [12, 1, 2]:      return 'Winter'
    elif month in [3, 4, 5]:     return 'Summer'
    elif month in [6, 7, 8, 9]:  return 'Monsoon'
    else:                         return 'Post-Monsoon'

df_merged['season'] = df_merged['mon'].apply(get_season)
df_merged['quarter'] = ((df_merged['mon'] - 1) // 3 + 1)

# Rolling 4-week average of cases per district+disease
df_merged = df_merged.sort_values(['district','disease','year','mon','day'])
df_merged['cases_4wk_avg'] = (
    df_merged.groupby(['district','disease'])['cases']
    .transform(lambda x: x.rolling(4, min_periods=1).mean())
    .round(2)
)

# Drop helper column
df_merged = df_merged.drop(columns=['district_key'])

print('✅ Features added: season, quarter, cases_4wk_avg')
df_merged[['district','disease','mon','season','cases','cases_4wk_avg']].head(8)

✅ Features added: season, quarter, cases_4wk_avg


,district,disease,mon,season,cases,cases_4wk_avg
119,Ariyalur,Acute Diarrhoeal Disease,4,Summer,6,6.00
128,Ariyalur,Acute Diarrhoeal Disease,6,Monsoon,9,7.50
74,Ariyalur,Acute Diarrhoeal Disease,4,Summer,11,8.67
5,Ariyalur,Acute Diarrhoeal Disease,3,Summer,14,10.00
8,Ariyalur,Acute Diarrhoeal Disease,4,Summer,17,12.75
64,Ariyalur,Chikungunya,2,Winter,15,15.00
609,Ariyalur,Dengue,12,Winter,7,7.00
101,Ariyalur,Dengue,1,Winter,12,9.50


In [12]:
# ── Clean district names — remove sub-district/merged entries ──────
valid_districts = [
    'Ariyalur', 'Chengalpattu', 'Chennai', 'Coimbatore', 'Cuddalore',
    'Dharmapuri', 'Dindigul', 'Erode', 'Kallakurichi', 'Kanchipuram',
    'Kanniyakumari', 'Karur', 'Krishnagiri', 'Madurai', 'Mayiladuthurai',
    'Nagapattinam', 'Namakkal', 'Nilgiris', 'Perambalur', 'Pudukkottai',
    'Ramanathapuram', 'Ranipet', 'Salem', 'Sivaganga', 'Tenkasi',
    'Thanjavur', 'Theni', 'Thoothukudi', 'Tiruchirappalli', 'Tirunelveli',
    'Tirupathur', 'Tiruppur', 'Tiruvallur', 'Tiruvannamalai', 'Tiruvarur',
    'Vellore', 'Villupuram', 'Virudhunagar'
]

# Also match common alternate spellings from EpiClim
spelling_fix = {
    'Pudukottai'       : 'Pudukkottai',
    'Kanyakumari'      : 'Kanniyakumari',
    'The Nilgiris'     : 'Nilgiris',
    'Tiruchirapalli'   : 'Tiruchirappalli',
    'Thoothukudi'      : 'Thoothukudi',
}
df_merged['district'] = df_merged['district'].replace(spelling_fix)

# Keep only valid districts
before = len(df_merged)
df_merged = df_merged[df_merged['district'].isin(valid_districts)].copy()
df_merged = df_merged.reset_index(drop=True)

print(f'Rows before clean : {before}')
print(f'Rows after clean  : {len(df_merged)}')
print(f'Districts kept    : {df_merged["district"].nunique()}')
print(sorted(df_merged['district'].unique()))

Rows before clean : 735
Rows after clean  : 632
Districts kept    : 32
['Ariyalur', 'Coimbatore', 'Cuddalore', 'Dharmapuri', 'Dindigul', 'Erode', 'Kallakurichi', 'Kanniyakumari', 'Karur', 'Krishnagiri', 'Madurai', 'Nagapattinam', 'Namakkal', 'Nilgiris', 'Perambalur', 'Pudukkottai', 'Ramanathapuram', 'Salem', 'Sivaganga', 'Tenkasi', 'Thanjavur', 'Theni', 'Thoothukudi', 'Tiruchirappalli', 'Tirunelveli', 'Tiruppur', 'Tiruvallur', 'Tiruvannamalai', 'Tiruvarur', 'Vellore', 'Villupuram', 'Virudhunagar']


In [13]:
SAVE_PATH = '/content/drive/MyDrive/DiseaseSpread/dataset/processed_data/merged_clean.csv'
df_merged.to_csv(SAVE_PATH, index=False)

print(f'✅ Processed data saved!')
print(f'Path  : {SAVE_PATH}')
print(f'Shape : {df_merged.shape}')
print(f'Final Columns: {df_merged.columns.tolist()}')

✅ Processed data saved!
Path  : /content/drive/MyDrive/DiseaseSpread/dataset/processed_data/merged_clean.csv
Shape : (632, 24)
Final Columns: ['Unnamed: 0', 'week_label', 'state', 'district', 'disease', 'cases', 'deaths', 'day', 'mon', 'year', 'latitude', 'longitude', 'rainfall_mm', 'leaf_area_index', 'Temp', 'temp_celsius', 'date', 'population', 'area_sq_km', 'density_per_sqkm', 'cases_per_100k', 'season', 'quarter', 'cases_4wk_avg']


## ✅ Preprocessing Summary

| Step | Action | Result |
|------|--------|--------|
| Filter | Tamil Nadu only | Focused dataset |
| Temp | Kelvin → Celsius | Readable values |
| Missing | Median imputation | 0 nulls |
| Census | Real 2011 district data | Population density added |
| Normalize | Cases per 100K | Comparable across districts |
| Features | Season, rolling avg | Ready for model |

> **Next Step →** `visualization.ipynb`